# Phase 4 — Comparaison de modèles et évaluation (FD001)

Objectif de ce notebook : répondre avec des **preuves chiffrées et reproductibles** à trois questions.

1. **XGBoost est-il justifié** face à des alternatives simples (référence naïve, Random Forest, LightGBM),
   sur la précision, la sécurité (surestimations) et le temps de calcul ? (section B)
2. **Peut-on rendre le modèle plus prudent** — moins de surestimations dangereuses — et à quel prix en
   précision ? D'abord la fausse piste (changer 13 et 10 dans la note), puis la bonne (pénaliser les
   surestimations dans l'apprentissage), puis une recherche systématique avec Optuna. (sections C et D)
3. **Que vaut le modèle retenu sur le jeu de test officiel**, évalué une seule fois, et que
   disent ses erreurs restantes ? (sections E et F)

Plan : A. Préparation et règles — B. Comparaison à réglages de base — C. Pénalité de surestimation —
D. Optuna et règle de choix — E. Évaluation finale sur le test — F. Moteurs surestimés — G. Synthèse.

Le calcul lourd (les études Optuna : 17 minutes avec `--parallele`, environ 37 minutes l'une après l'autre)
est fait par `scripts/lancer_optuna.py`, qui écrit
ses résultats dans `docs/resultats/`. Ce notebook **relit** ces résultats (`RELANCER_OPTUNA = False`
ci-dessous) : il s'exécute de bout en bout en quelques minutes.

## A. Préparation

### A.1 Règles méthodologiques (fixées avant tout résultat)

1. **Le jeu de test officiel ne sert jamais à choisir quoi que ce soit** — ni les features, ni le modèle,
   ni les réglages, ni un seuil. Tout choix se fait sur la validation croisée. Pourquoi ? Si on choisit
   en regardant le test, le test devient une donnée d'entraînement déguisée et son score devient
   optimiste : on ne saurait plus ce que vaut le modèle sur des moteurs vraiment nouveaux.
2. **Les mêmes 5 plis pour tous les modèles** : `GroupKFold` par moteur (un moteur entier est soit en
   entraînement, soit en validation), mélangé avec la graine 42, comme en Phase 3. Calculés **une seule
   fois** par `calculer_plis` et réutilisés partout : deux modèles sont donc comparés sur exactement les
   mêmes moteurs de validation, et l'écart observé ne vient pas d'un découpage différent.
3. **La règle de choix du modèle final est écrite avant de regarder le test** (section D, et codée et
   testée dans `src/optimisation.py`). Le test est évalué à la fin, une seule fois, pour les modèles candidats.
4. **Le score NASA d'évaluation garde toujours ses constantes officielles 13 / 10.** Ce qu'on fait varier,
   c'est la perte d'**apprentissage** (section C).
5. **Statuts honnêtes** : un écart non significatif ou un résultat mixte est dit comme tel.

### A.2 Imports et réglages

On réutilise les modules de `src/` (déjà testés par pytest) au lieu de recopier le pipeline :
`features.py` (préparation Phases 1-3), `evaluation.py` (métriques, validation croisée, tests appariés),
`modeles.py` (fabriques de modèles et pertes sur mesure), `config_modele.py` (description d'un modèle
par un petit dictionnaire, partagé avec la démo) et `optimisation.py` (Optuna, front de Pareto, règle de choix).

In [ ]:
import sys
sys.path.insert(0, "..")  # pour pouvoir importer src/ depuis notebooks/

import warnings
warnings.filterwarnings("ignore", message="IProgress not found")  # avertissement d'affichage sans effet (barre de progression)

import copy
import datetime
import json
import os
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import stats

from src.config_modele import CONFIG_PHASE3, construire_fabrique_depuis_config, decrire_config, sauvegarder_config
from src.evaluation import (
    calculer_plis,
    comparer_apparie,
    demontrer_constantes_score_degenerees,
    erreurs_par_moteur,
    evaluer_en_validation_croisee,
    evaluer_sur_test,
    nb_surestimes,
    rmse,
    score_nasa,
)
from src.features import preparer_test, preparer_train
from src.optimisation import (
    DOSSIER_RESULTATS,
    N_ESSAIS_PAR_DEFAUT,
    NOMS_ETUDES,
    TEXTE_REGLE_DE_CHOIX,
    TOLERANCE_RMSE,
    appliquer_regle_de_choix,
    calculer_front_pareto,
    charger_essais,
    chemin_csv_etude,
    config_depuis_ligne,
    lancer_etude,
)

# False : on relit les CSV de docs/resultats/ ecrits par scripts/lancer_optuna.py (quelques secondes).
# True  : on relance les 3 etudes Optuna ici, l'une apres l'autre (environ 40 minutes) et on reecrit les CSV.
RELANCER_OPTUNA = False

SEUIL_SURESTIMATION = 20  # cycles : au-dela, une surestimation est comptee comme dangereuse (choix Phase 3)
SEUIL_ALERTE = 30         # cycles : seuil d'envoi en maintenance, HYPOTHESE DE TRAVAIL (pas de valeur officielle)
PLAFOND_RUL = 125         # plafond de la RUL (Phase 2)

DOSSIER_FIGURES = os.path.join("..", "docs", "figures")  # figures versionnees du projet
os.makedirs(DOSSIER_FIGURES, exist_ok=True)
os.makedirs(DOSSIER_RESULTATS, exist_ok=True)  # docs/resultats/ (CSV et configuration retenue)

# couleurs des figures : 3 teintes distinctes (lisibles aussi par les daltoniens), toujours la meme
# teinte pour la meme chose d'une figure a l'autre ; le texte et les reperes restent en gris / noir
COULEUR_1 = "#2a78d6"  # bleu
COULEUR_2 = "#eb6834"  # orange
COULEUR_3 = "#1baf7a"  # vert d'eau
COULEUR_TEXTE = "#52514e"  # gris fonce (reperes, annotations)
plt.rcParams["axes.grid"] = True
plt.rcParams["axes.axisbelow"] = True  # quadrillage dessine sous les barres et les points
plt.rcParams["grid.color"] = "#e4e3df"  # quadrillage discret
plt.rcParams["axes.spines.top"] = False
plt.rcParams["axes.spines.right"] = False

pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 40)
debut_notebook = time.perf_counter()  # chronometre de l'execution complete du notebook

### A.3 Données d'entraînement et de test

`preparer_train` enchaîne exactement le pipeline de la Phase 3 (capteurs constants retirés, min-max appris
sur le train, lissage, RUL plafonnée à 125, features glissantes) : 20 631 lignes, 46 features.
`preparer_test` applique les mêmes transformations au test **sans rien réapprendre** (même scaler en
`transform` seul) et garde le dernier cycle observé de chaque moteur test, comparé à `RUL_FD001.txt`.

In [ ]:
contexte_train = preparer_train(fenetre=5, plafond=PLAFOND_RUL)  # pipeline Phases 1 a 3 sur le train
contexte_test = preparer_test(contexte_train)  # memes etapes sur le test, sans fuite

X = contexte_train["X"]            # 46 features, une ligne par cycle
y = contexte_train["y"]            # cible : RUL plafonnee a 125
groups = contexte_train["groups"]  # numero de moteur de chaque ligne (pour GroupKFold)

X_test_dernier = contexte_test["X_test_dernier"]  # dernier cycle observe de chaque moteur test
y_test_vraie = contexte_test["y_test_vraie"]      # RUL vraie BRUTE de RUL_FD001.txt (non plafonnee)

print(f"Train : {X.shape[0]} lignes, {X.shape[1]} features, {groups.nunique()} moteurs")
print(f"Test  : {X_test_dernier.shape[0]} moteurs (un point par moteur, au dernier cycle observe)")
assert X.shape == (20631, 46)
assert X_test_dernier.shape == (100, 46)

### A.4 Les plis de validation croisée, calculés une seule fois

`calculer_plis` vérifie lui-même qu'aucun moteur n'est des deux côtés d'un pli et que chaque moteur est en
validation exactement une fois. Ces 5 plis servent à **tous** les modèles de ce notebook et à tous les
essais Optuna (le script recalcule exactement les mêmes : mêmes données, même graine).

In [ ]:
plis = calculer_plis(X, y, groups, n_splits=5, random_state=42)  # memes reglages que la Phase 3

NB_MOTEURS_TRAIN = int(groups.nunique())  # 100 moteurs d'entrainement
for numero_pli, (idx_train, idx_val) in enumerate(plis):  # un pli a la fois
    nb_moteurs_val = groups.iloc[idx_val].nunique()  # moteurs mis de cote dans ce pli
    print(f"Pli {numero_pli + 1} : {len(idx_train)} lignes d'entrainement, {len(idx_val)} lignes de validation "
          f"({nb_moteurs_val} moteurs)")

### A.5 Deux petites fonctions de mise en forme

`evaluer_en_validation_croisee` renvoie beaucoup de chiffres. `resumer_cv` en extrait une ligne de tableau,
toujours avec les mêmes colonnes, pour que tous les tableaux de ce notebook se lisent de la même façon :

- **RMSE CV moyen ± écart-type entre plis** : la précision, et sa variabilité d'un groupe de moteurs à l'autre ;
- **MAE** : erreur absolue moyenne (moins sensible aux grosses erreurs isolées que le RMSE) ;
- **score NASA CV moyen** : moyenne sur les 5 plis du score NASA (13/10) de chaque pli — calculé sur
  toutes les lignes de validation (environ 4 000 par pli), d'où des valeurs de l'ordre de 30 000 (le score
  du test, lui, ne porte que sur 100 points) ;
- **surestimés > 20 (OOF)** : nombre de lignes, parmi les 20 631 prédictions hors-pli (OOF = *out-of-fold* :
  chaque ligne est prédite par le modèle du pli où elle était en validation), surestimées de plus de 20 cycles ;
- **alertes manquées / fausses alertes (OOF)** : avec un seuil d'envoi en maintenance à 30 cycles
  (hypothèse de travail) — alerte manquée = RUL vraie ≤ 30 mais prédiction > 30 (dangereux) ;
  fausse alerte = RUL vraie > 30 mais prédiction ≤ 30 (coûteux, pas dangereux) ;
- **biais OOF** : moyenne de (prédiction − vérité) ; négatif = le modèle sous-estime en moyenne (prudent) ;
- **temps** : entraînement cumulé des 5 plis (s), et prédiction en millisecondes **par moteur** (toute la
  trajectoire d'un moteur, environ 206 cycles, prédite d'un coup).

`resumer_test` fait de même pour le jeu de test, et `metriques_par_moteur` calcule, moteur par moteur, le
RMSE, la pénalité NASA (somme des pénalités des cycles du moteur) et le nombre de cycles surestimés de plus
de 20 : ce sont les valeurs **appariées** dont les tests statistiques ont besoin.

In [ ]:
def resumer_cv(nom, resultats, nb_moteurs):
    """
    Resume les resultats d'une validation croisee en une ligne de tableau (un dictionnaire).

    Parameters
    ----------
    nom : str
        Le nom du modele (premiere colonne du tableau).
    resultats : dict
        Le dictionnaire renvoye par `evaluer_en_validation_croisee`.
    nb_moteurs : int
        Nombre de moteurs predits au total (100), pour le temps de prediction par moteur.

    Returns
    -------
    dict
        Les colonnes communes a tous les tableaux de validation croisee de ce notebook.
    """
    ligne = {}
    ligne["modele"] = nom
    ligne["rmse_cv_moyen"] = resultats["rmse_moyen"]
    ligne["rmse_cv_ecart_type"] = resultats["rmse_ecart_type"]
    ligne["mae_cv_moyen"] = resultats["mae_moyen"]
    ligne["nasa_cv_moyen"] = resultats["nasa_moyen"]
    ligne["surestimes_oof"] = resultats["surestimes_oof"]
    ligne["alertes_manquees_oof"] = resultats["alertes_manquees_oof"]
    ligne["fausses_alertes_oof"] = resultats["fausses_alertes_oof"]
    ligne["biais_oof"] = resultats["biais_oof"]
    ligne["temps_entrainement_5_plis_s"] = resultats["temps_entrainement_total"]
    # temps de prediction total des 5 plis (secondes) ramene a un moteur, en millisecondes
    ligne["prediction_ms_par_moteur"] = 1000.0 * resultats["temps_prediction_total"] / nb_moteurs
    return ligne


def resumer_test(nom, resultats_test):
    """
    Resume l'evaluation d'un modele sur le test officiel en une ligne de tableau (un dictionnaire).

    Parameters
    ----------
    nom : str
        Le nom du modele.
    resultats_test : dict
        Le dictionnaire renvoye par `evaluer_sur_test`.

    Returns
    -------
    dict
        RMSE (contre la RUL brute et contre la RUL plafonnee), MAE, score NASA, surestimes > 20,
        alertes manquees et fausses alertes, sur les 100 moteurs test.
    """
    ligne = {}
    ligne["modele"] = nom
    ligne["rmse_test"] = resultats_test["rmse"]
    ligne["rmse_test_rul_plafonnee"] = resultats_test["rmse_contre_rul_plafonnee"]
    ligne["mae_test"] = resultats_test["mae"]
    ligne["nasa_test"] = resultats_test["score_nasa"]
    ligne["surestimes_test"] = resultats_test["nb_surestimes"]
    ligne["alertes_manquees_test"] = resultats_test["alertes_manquees"]
    ligne["fausses_alertes_test"] = resultats_test["fausses_alertes"]
    return ligne


def metriques_par_moteur(y_true, y_pred, groups):
    """
    Calcule trois metriques moteur par moteur : RMSE, penalite NASA (13/10) et surestimations > 20.

    Parameters
    ----------
    y_true : array-like
        Les vraies valeurs (une par ligne).
    y_pred : array-like
        Les predictions (une par ligne, meme ordre).
    groups : array-like
        Le numero de moteur de chaque ligne (sur le test : une seule ligne par moteur).

    Returns
    -------
    pandas.DataFrame
        Une ligne par moteur (triee par numero) : "unit_number", "rmse", "nasa" (somme des penalites
        NASA des lignes du moteur) et "surestimes" (lignes surestimees de plus de 20 cycles).
    """
    vrais = np.asarray(y_true, dtype=float)
    predits = np.asarray(y_pred, dtype=float)
    groupes = np.asarray(groups)

    lignes = []
    for moteur_id in np.unique(groupes):  # un moteur a la fois, dans l'ordre croissant
        masque = groupes == moteur_id  # Vrai sur les lignes de ce moteur
        ligne = {}
        ligne["unit_number"] = moteur_id
        ligne["rmse"] = rmse(vrais[masque], predits[masque])
        ligne["nasa"] = score_nasa(vrais[masque], predits[masque])
        ligne["surestimes"] = nb_surestimes(vrais[masque], predits[masque], seuil=SEUIL_SURESTIMATION)
        lignes.append(ligne)
    return pd.DataFrame(lignes)

## B. Comparaison à réglages de base, sur les mêmes plis

Quatre modèles, chacun avec des réglages « raisonnables par défaut » (aucun n'est optimisé ici, pour une
comparaison équitable) :

| Modèle | Réglages |
|---|---|
| Référence naïve | prédit toujours la moyenne de la cible d'entraînement : la borne basse |
| Random Forest | 200 arbres, profondeur non bornée (défauts scikit-learn) |
| XGBoost Phase 3 | 200 arbres, profondeur 5, pas 0,05 (exactement la Phase 3) |
| LightGBM | 200 arbres, pas 0,05, 31 feuilles par arbre (défauts LightGBM), mode déterministe |

Chaque modèle est décrit par un petit dictionnaire (format de `src/config_modele.py`, celui que lit aussi
la démo). `n_jobs` (nombre de cœurs) ne change pas les résultats — vérifié : RMSE identique au bit près
avec 4, 15 ou tous les cœurs — seulement la vitesse.

**Contrôle de reproduction** : XGBoost Phase 3 doit redonner, dans cet environnement, CV RMSE moyen 16,87
et score NASA CV moyen 29 782,6 (le document de Phase 3 écrit 16,86 / 29 729,3, obtenus dans un autre
environnement : écart d'environnement déjà vérifié, voir la note de reproductibilité de la Phase 3).

In [ ]:
# configuration de chaque modele : famille, perte, parametres de la perte, hyperparametres
CONFIG_REFERENCE_NAIVE = {
    "famille": "reference_naive", "perte": "native", "parametres_perte": {}, "hyperparametres": {},
    "nom": "Reference naive (moyenne)",
}
CONFIG_RANDOM_FOREST = {
    "famille": "random_forest", "perte": "native", "parametres_perte": {},
    "hyperparametres": {"n_estimators": 200, "random_state": 42, "n_jobs": -1},
    "nom": "Random Forest (base)",
}
CONFIG_XGB_PHASE3 = copy.deepcopy(CONFIG_PHASE3)  # exactement la Phase 3 (copie : l'original reste intact)
CONFIG_LIGHTGBM_BASE = {
    "famille": "lightgbm", "perte": "native", "parametres_perte": {},
    "hyperparametres": {"n_jobs": 4},  # le reste : valeurs de fabrique_lightgbm (200 arbres, pas 0.05, ...)
    "nom": "LightGBM (base)",
}

CONFIGS_BASE = {}
CONFIGS_BASE["Reference naive"] = CONFIG_REFERENCE_NAIVE
CONFIGS_BASE["Random Forest"] = CONFIG_RANDOM_FOREST
CONFIGS_BASE["XGBoost Phase 3"] = CONFIG_XGB_PHASE3
CONFIGS_BASE["LightGBM"] = CONFIG_LIGHTGBM_BASE

resultats_cv_base = {}  # nom du modele -> dictionnaire de evaluer_en_validation_croisee
for nom_modele in CONFIGS_BASE:  # un modele a la fois, toujours les memes plis
    fabrique = construire_fabrique_depuis_config(CONFIGS_BASE[nom_modele])  # fonction -> modele neuf
    debut = time.perf_counter()
    resultats_cv_base[nom_modele] = evaluer_en_validation_croisee(
        fabrique, X, y, groups, plis, seuil_surestimation=SEUIL_SURESTIMATION, seuil_alerte=SEUIL_ALERTE
    )
    duree = time.perf_counter() - debut
    print(f"{nom_modele:16s} : RMSE CV moyen = {resultats_cv_base[nom_modele]['rmse_moyen']:.2f} ({duree:.1f} s)")

# controle de reproduction de la Phase 3 dans cet environnement
assert round(resultats_cv_base["XGBoost Phase 3"]["rmse_moyen"], 2) == 16.87
assert round(resultats_cv_base["XGBoost Phase 3"]["nasa_moyen"], 1) == 29782.6

### B.1 Tableau comparatif (validation croisée)

In [ ]:
lignes = []
for nom_modele in resultats_cv_base:
    lignes.append(resumer_cv(nom_modele, resultats_cv_base[nom_modele], NB_MOTEURS_TRAIN))
tableau_cv_base = pd.DataFrame(lignes).set_index("modele")
tableau_cv_base.round(2)

### B.2 Les écarts sont-ils significatifs ? Tests appariés par moteur

Comparer deux RMSE moyens ne suffit pas : une différence de 0,1 cycle peut n'être que du bruit. On calcule
donc, pour chaque modèle, le **RMSE hors-pli de chacun des 100 moteurs**, et on compare moteur par moteur
avec XGBoost Phase 3 (`comparer_apparie`) :

- **test t apparié** : la moyenne des différences (modèle − XGBoost) est-elle nulle ?
- **test de Wilcoxon signé** : même question, par les rangs, sans supposer de loi normale.

« Apparié » : les deux modèles sont jugés sur les **mêmes** moteurs, donc la difficulté propre à chaque
moteur s'annule dans la différence. Une différence moyenne **négative** veut dire que le modèle a un RMSE
par moteur plus petit que XGBoost (il est meilleur). On fait 3 comparaisons : pour garder un risque global
de fausse découverte de 5 %, on lit chaque p-valeur contre 0,05 / 3 ≈ 0,017 (correction de Bonferroni,
volontairement prudente).

In [ ]:
# RMSE hors-pli de chaque moteur, pour chaque modele (100 lignes, triees par numero de moteur)
erreurs_moteur_base = {}
for nom_modele in resultats_cv_base:
    erreurs_moteur_base[nom_modele] = erreurs_par_moteur(y, resultats_cv_base[nom_modele]["predictions_oof"], groups)

rmse_moteur_xgb = erreurs_moteur_base["XGBoost Phase 3"]["rmse"].to_numpy()  # reference des comparaisons
SEUIL_BONFERRONI = 0.05 / 3  # 3 comparaisons contre XGBoost Phase 3

lignes = []
for nom_modele in resultats_cv_base:
    if nom_modele == "XGBoost Phase 3":
        continue  # on ne compare pas XGBoost a lui-meme
    rmse_moteur_modele = erreurs_moteur_base[nom_modele]["rmse"].to_numpy()
    comparaison = comparer_apparie(rmse_moteur_modele, rmse_moteur_xgb)
    ligne = {}
    ligne["modele (vs XGBoost Phase 3)"] = nom_modele
    ligne["n_moteurs"] = comparaison["n"]
    ligne["difference_moyenne_rmse_par_moteur"] = comparaison["difference_moyenne"]
    ligne["moteurs_ou_le_modele_est_meilleur"] = int(np.sum(rmse_moteur_modele < rmse_moteur_xgb))
    ligne["p_valeur_t"] = comparaison["p_valeur_t"]
    ligne["p_valeur_wilcoxon"] = comparaison["p_valeur_wilcoxon"]
    ligne["significatif_bonferroni"] = (comparaison["p_valeur_t"] < SEUIL_BONFERRONI) and (
        comparaison["p_valeur_wilcoxon"] < SEUIL_BONFERRONI)
    lignes.append(ligne)
tableau_tests_base = pd.DataFrame(lignes).set_index("modele (vs XGBoost Phase 3)")
tableau_tests_base.round(4)

### B.3 Figure de comparaison

Quatre panneaux, un critère par panneau (jamais deux échelles sur un même axe). La référence naïve est
hors échelle (RMSE autour de 41) : elle est indiquée dans le titre plutôt que dessinée. Panneau RMSE :
la barre est la moyenne des 5 plis, les points sont les 5 plis.

In [ ]:
noms_compares = ["Random Forest", "XGBoost Phase 3", "LightGBM"]  # la reference naive est hors echelle
positions = np.arange(len(noms_compares))

fig, axes = plt.subplots(2, 2, figsize=(12, 8))

# (1) RMSE CV : moyenne (barre) et valeur de chaque pli (points)
ax = axes[0, 0]
for position, nom_modele in zip(positions, noms_compares):
    resultats = resultats_cv_base[nom_modele]
    ax.bar(position, resultats["rmse_moyen"], width=0.6, color=COULEUR_1, alpha=0.85)
    ax.scatter(np.full(5, position), resultats["rmse_par_pli"], color="black", s=18, zorder=3)
    ax.text(position, 19.5, f"moyenne {resultats['rmse_moyen']:.2f}", ha="center", fontsize=9)  # au-dessus des points
ax.set_ylim(14, 20)
ax.set_ylabel("RMSE (cycles)")
ax.set_title("RMSE CV moyen (barre) et par pli (points)")

# (2) score NASA CV moyen
ax = axes[0, 1]
for position, nom_modele in zip(positions, noms_compares):
    valeur = resultats_cv_base[nom_modele]["nasa_moyen"]
    ax.bar(position, valeur, width=0.6, color=COULEUR_1, alpha=0.85)
    ax.text(position, valeur * 1.01, f"{valeur:,.0f}".replace(",", " "), ha="center", fontsize=9)
ax.set_ylabel("score NASA (13/10), moyenne des 5 plis")
ax.set_title("Score NASA CV moyen (plus bas = mieux)")

# (3) surestimations dangereuses : lignes OOF surestimees de plus de 20 cycles
ax = axes[1, 0]
for position, nom_modele in zip(positions, noms_compares):
    valeur = resultats_cv_base[nom_modele]["surestimes_oof"]
    ax.bar(position, valeur, width=0.6, color=COULEUR_2, alpha=0.85)
    ax.text(position, valeur * 1.01, str(valeur), ha="center", fontsize=9)
ax.set_ylabel("lignes OOF (sur 20 631)")
ax.set_title("Surestimés de plus de 20 cycles (OOF)")

# (4) temps d'entrainement cumule sur les 5 plis
ax = axes[1, 1]
for position, nom_modele in zip(positions, noms_compares):
    valeur = resultats_cv_base[nom_modele]["temps_entrainement_total"]
    ax.bar(position, valeur, width=0.6, color=COULEUR_TEXTE, alpha=0.85)
    ax.text(position, valeur * 1.02, f"{valeur:.1f} s", ha="center", fontsize=9)
ax.set_ylabel("secondes (5 plis)")
ax.set_title("Temps d'entraînement (5 plis)")

for ax in axes.ravel():  # memes etiquettes de modeles sous chaque panneau
    ax.set_xticks(positions)
    ax.set_xticklabels(noms_compares)

rmse_naive = resultats_cv_base["Reference naive"]["rmse_moyen"]
fig.suptitle(f"Phase 4 — comparaison à réglages de base, mêmes 5 plis GroupKFold "
             f"(référence naïve hors échelle : RMSE CV = {rmse_naive:.1f})")
plt.tight_layout()
plt.savefig(os.path.join(DOSSIER_FIGURES, "04_comparaison_modeles.png"), dpi=150)
plt.show()

### B.4 Conclusion de la section B

- **Les modèles apprennent** : la référence naïve a un RMSE CV de 41,7 cycles, les trois ensembles d'arbres
  environ 16,9 (test apparié : p < 0,0001).
- **Précision : égalité statistique** entre Random Forest (16,96), XGBoost Phase 3 (16,87) et LightGBM
  (16,85). Les écarts (0,11 cycle au plus) sont dix fois plus petits que la variation d'un pli à l'autre
  (écart-type ≈ 1,3), et les tests appariés par moteur ne détectent rien (Random Forest : p = 0,78 / 0,58 ;
  LightGBM : p = 0,79 / 0,80).
- **Sécurité : même constat.** Surestimés OOF > 20 : 2 302 (Random Forest), 2 384 (XGBoost), 2 316
  (LightGBM) ; alertes manquées 565 / 552 / 568 ; score NASA CV 29 773 / 29 783 / 30 142. Aucun modèle
  n'est nettement plus prudent qu'un autre à réglages de base.
- **Temps : XGBoost est le plus rapide** sur cette machine : environ 4 s d'entraînement pour les 5 plis,
  contre environ 7 s pour LightGBM et 40 s pour la Random Forest (qui utilise pourtant tous les cœurs) ; il
  prédit la trajectoire complète d'un moteur en moins d'une milliseconde, contre environ 5 ms pour la forêt.

**Réponse honnête à la question « XGBoost est-il justifié ? »** : XGBoost n'est **pas plus précis** que
les alternatives, il est à égalité. Son choix se justifie par des critères pratiques : il est environ 9 fois
plus rapide à entraîner que la Random Forest (utile pour une recherche Optuna de plus de 100 essais) et,
contrairement à la Random Forest, il accepte une perte d'apprentissage sur mesure, dont la section C a
besoin. LightGBM a les mêmes atouts : il est donc gardé dans la recherche Optuna (section D).

## C. Rendre le modèle plus prudent : où mettre la pénalité ?

### C.1 Fausse piste : changer 13 et 10 dans la note ne change aucune prédiction

L'idée de départ (coach) : « faire varier 10 et 13 avec Optuna ». Si on fait varier les constantes du
**score NASA**, on change seulement la *note* donnée à des prédictions **déjà faites** : le modèle n'est
même pas consulté. Démonstration sur les prédictions test de XGBoost Phase 3 — on regarde le test ici
uniquement pour une démonstration mathématique, **pas pour choisir quoi que ce soit** : on multiplie les
deux dénominateurs par un facteur k.

In [ ]:
# modele Phase 3 entraine sur les 100 moteurs (le meme qu'en Phase 3), predictions des 100 moteurs test
modele_phase3_demo = construire_fabrique_depuis_config(CONFIG_XGB_PHASE3)()
modele_phase3_demo.fit(X, y)
predictions_test_phase3_demo = np.clip(modele_phase3_demo.predict(X_test_dernier), a_min=0, a_max=None)

tableau_constantes = demontrer_constantes_score_degenerees(y_test_vraie, predictions_test_phase3_demo,
                                                          facteurs=(1, 2, 5, 10, 100, 1000))
tableau_constantes["rmse_test"] = rmse(y_test_vraie, predictions_test_phase3_demo)  # les predictions ne changent pas : RMSE identique
tableau_constantes.round(3)

Lecture : les prédictions (et donc le RMSE) sont **identiques** sur toutes les lignes, seule la note change ;
et plus on agrandit les dénominateurs, plus le score tend vers 0 — pour n'importe quel modèle, même mauvais.
« Optimiser » ces constantes reviendrait donc à pousser k vers l'infini : la note ne mesurerait plus rien.

### C.2 La bonne version : la pénalité dans la perte d'apprentissage

Pour que le modèle **prédise** autrement, il faut changer ce qu'il minimise pendant l'entraînement (sa
« perte »). Deux pertes sur mesure existent dans `src/modeles.py` (testées par pytest) :

- **quadratique asymétrique** : erreur² × α si le modèle surestime, erreur² × 1 sinon. α = 1 redonne la
  perte classique ; α = 2 rend une surestimation deux fois plus coûteuse qu'une sous-estimation de même taille.
- **forme NASA** : exp(d / h_sur) − 1 si d ≥ 0, exp(−d / h_sous) − 1 si d < 0 (d = prédiction − vérité),
  avec h_sous = 13 et h_sur = 10 pour reprendre la forme du score, ou h_sur = 5 pour être plus prudent.

La note d'évaluation reste le score NASA 13/10 dans tous les cas.

Petite grille explicative, sur les mêmes plis, en gardant les réglages Phase 3 (200 arbres, profondeur 5,
pas 0,05) sauf mention contraire :

- XGBoost et LightGBM, perte quadratique asymétrique, α ∈ {1 ; 1,5 ; 2 ; 3} ;
- XGBoost, perte NASA (13, 10) et (13, 5), avec 200 arbres (réglages Phase 3), puis avec **1 000 et
  2 000 arbres au pas 0,1** pour vérifier la convergence. Pourquoi ? Avec cette perte, chaque arbre ne
  déplace une feuille que d'environ « pas × h » cycles (diagnostic des docstrings de `src/modeles.py`) :
  avec h = 5 et le pas 0,05, c'est 0,25 cycle par arbre, et 200 arbres peuvent ne pas suffire. Si 1 000 et
  2 000 arbres donnent presque les mêmes chiffres, l'apprentissage a convergé.

In [ ]:
reglages_penalite = []  # une entree par reglage : nom lisible, groupe (pour la figure) et configuration

for alpha in [1.0, 1.5, 2.0, 3.0]:  # XGBoost, perte quadratique asymetrique, reglages Phase 3
    reglage = {}
    reglage["reglage"] = f"XGBoost quadratique alpha={alpha:g}"
    reglage["groupe"] = "XGBoost quadratique asymétrique"
    reglage["config"] = {"famille": "xgboost", "perte": "quadratique_asymetrique",
                         "parametres_perte": {"alpha": alpha}, "hyperparametres": {"n_jobs": 4}}
    reglages_penalite.append(reglage)

for alpha in [1.0, 1.5, 2.0, 3.0]:  # LightGBM, perte quadratique asymetrique, reglages de base
    reglage = {}
    reglage["reglage"] = f"LightGBM quadratique alpha={alpha:g}"
    reglage["groupe"] = "LightGBM quadratique asymétrique"
    reglage["config"] = {"famille": "lightgbm", "perte": "quadratique_asymetrique",
                         "parametres_perte": {"alpha": alpha}, "hyperparametres": {"n_jobs": 4}}
    reglages_penalite.append(reglage)

for h_sous, h_sur in [(13.0, 10.0), (13.0, 5.0)]:  # XGBoost, perte de forme NASA
    for n_arbres, pas in [(200, 0.05), (1000, 0.1), (2000, 0.1)]:
        reglage = {}
        reglage["reglage"] = f"XGBoost NASA ({h_sous:g},{h_sur:g}) {n_arbres} arbres pas {pas:g}"
        if n_arbres == 200:
            reglage["groupe"] = "XGBoost perte NASA, 200 arbres"
        else:
            reglage["groupe"] = "XGBoost perte NASA, 1000-2000 arbres"
        reglage["config"] = {"famille": "xgboost", "perte": "nasa",
                             "parametres_perte": {"h_sous": h_sous, "h_sur": h_sur},
                             "hyperparametres": {"n_estimators": n_arbres, "learning_rate": pas, "n_jobs": 4}}
        reglages_penalite.append(reglage)

resultats_cv_penalite = {}  # nom du reglage -> resultats de la validation croisee
for reglage in reglages_penalite:
    fabrique = construire_fabrique_depuis_config(reglage["config"])
    debut = time.perf_counter()
    resultats_cv_penalite[reglage["reglage"]] = evaluer_en_validation_croisee(
        fabrique, X, y, groups, plis, seuil_surestimation=SEUIL_SURESTIMATION, seuil_alerte=SEUIL_ALERTE
    )
    print(f"{reglage['reglage']:45s} ({time.perf_counter() - debut:5.1f} s)")

Tableau de la grille (la ligne « XGBoost Phase 3 (perte native) » est la référence de la section B) :

In [ ]:
lignes = []
ligne_reference = resumer_cv("XGBoost Phase 3 (perte native)", resultats_cv_base["XGBoost Phase 3"], NB_MOTEURS_TRAIN)
ligne_reference["groupe"] = "référence"
lignes.append(ligne_reference)
for reglage in reglages_penalite:
    ligne = resumer_cv(reglage["reglage"], resultats_cv_penalite[reglage["reglage"]], NB_MOTEURS_TRAIN)
    ligne["groupe"] = reglage["groupe"]
    lignes.append(ligne)
tableau_penalite = pd.DataFrame(lignes).set_index("modele")

colonnes_affichees = ["rmse_cv_moyen", "rmse_cv_ecart_type", "nasa_cv_moyen", "surestimes_oof",
                      "alertes_manquees_oof", "fausses_alertes_oof", "biais_oof", "temps_entrainement_5_plis_s"]
tableau_penalite[colonnes_affichees].round(2)

**Diagnostic de convergence de la perte NASA** : on compare 1 000 et 2 000 arbres (pas 0,1). Si les
chiffres bougent peu, l'apprentissage a convergé ; s'ils bougent beaucoup entre 200 et 1 000 arbres, les
200 arbres de la Phase 3 n'avaient pas fini d'apprendre.

In [ ]:
for h_sous, h_sur in [(13, 10), (13, 5)]:
    print(f"Perte NASA ({h_sous},{h_sur}) :")
    for n_arbres, pas in [(200, 0.05), (1000, 0.1), (2000, 0.1)]:
        nom_reglage = f"XGBoost NASA ({h_sous:g},{h_sur:g}) {n_arbres} arbres pas {pas:g}"
        resultats = resultats_cv_penalite[nom_reglage]
        print(f"  {n_arbres:5d} arbres, pas {pas:4.2f} : RMSE CV = {resultats['rmse_moyen']:.2f}, "
              f"NASA CV = {resultats['nasa_moyen']:8.1f}, surestimes OOF > 20 = {resultats['surestimes_oof']:4d}, "
              f"biais OOF = {resultats['biais_oof']:+.2f}")

### C.3 Figure : le compromis précision / prudence

Un point par réglage. Axe horizontal : RMSE CV moyen (précision, plus à gauche = mieux). Axe vertical :
surestimés OOF > 20 (panneau de gauche) ou score NASA CV moyen (panneau de droite), plus bas = plus prudent.
Les points d'une même famille sont reliés dans l'ordre d'α croissant.

In [ ]:
# (groupe, couleur, marqueur, style de ligne, decalage de l'etiquette en points, alignement horizontal) :
# les etiquettes XGBoost vont a gauche des points, celles de LightGBM a droite, pour ne pas se chevaucher
groupes_figure = [
    ("XGBoost quadratique asymétrique", COULEUR_1, "o", "-", (-7, -11), "right"),
    ("LightGBM quadratique asymétrique", COULEUR_2, "s", "-", (7, 3), "left"),
    ("XGBoost perte NASA, 200 arbres", COULEUR_3, "D", "none", (7, -12), "left"),
    ("XGBoost perte NASA, 1000-2000 arbres", COULEUR_3, "^", "none", (7, -10), "left"),
]

fig, axes = plt.subplots(1, 2, figsize=(15, 6.5))
for ax, colonne_y, titre_y in [(axes[0], "surestimes_oof", "surestimés > 20 cycles (lignes OOF)"),
                                (axes[1], "nasa_cv_moyen", "score NASA CV moyen (13/10)")]:
    for groupe, couleur, marqueur, style_ligne, decalage, alignement in groupes_figure:
        sous_tableau = tableau_penalite[tableau_penalite["groupe"] == groupe]
        ax.plot(sous_tableau["rmse_cv_moyen"], sous_tableau[colonne_y], marker=marqueur, linestyle=style_ligne,
                color=couleur, markersize=8, linewidth=2, label=groupe)
        for nom_reglage in sous_tableau.index:  # etiquette courte a cote de chaque point
            etiquette = nom_reglage.replace("XGBoost quadratique ", "").replace("LightGBM quadratique ", "")
            etiquette = etiquette.replace("XGBoost NASA ", "").replace(" arbres pas ", " arb. pas ")
            decalage_point = decalage
            if "2000 arbres" in nom_reglage:  # 1000 et 2000 arbres sont tres proches : etiquette au-dessus
                decalage_point = (7, 4)
            ax.annotate(etiquette, (sous_tableau.loc[nom_reglage, "rmse_cv_moyen"], sous_tableau.loc[nom_reglage, colonne_y]),
                        textcoords="offset points", xytext=decalage_point, ha=alignement, fontsize=7, color=COULEUR_TEXTE)
    reference = tableau_penalite.loc["XGBoost Phase 3 (perte native)"]
    ax.scatter(reference["rmse_cv_moyen"], reference[colonne_y], marker="*", s=250, color="black", zorder=4,
               label="XGBoost Phase 3 (perte native)")
    ax.set_xlim(16.0, 20.6)  # marge a gauche pour les etiquettes XGBoost
    ax.set_xlabel("RMSE CV moyen (cycles) — plus à gauche = plus précis")
    ax.set_ylabel(titre_y + " — plus bas = plus prudent")
axes[0].set_title("Précision vs surestimations dangereuses")
axes[1].set_title("Précision vs score NASA")
poignees, etiquettes = axes[0].get_legend_handles_labels()
fig.legend(poignees, etiquettes, loc="lower center", ncol=5, fontsize=9, frameon=False)  # legende commune, sous les panneaux
fig.suptitle("Pénaliser la surestimation dans l'apprentissage : pas de gain gratuit (validation croisée, mêmes 5 plis)")
plt.tight_layout(rect=(0, 0.06, 1, 1))  # on laisse la place de la legende en bas
plt.savefig(os.path.join(DOSSIER_FIGURES, "04_penalite_surestimation.png"), dpi=150)
plt.show()

### C.4 Conclusion de la section C

1. **Changer 13 et 10 dans la note ne change rien au modèle** : sur les 100 moteurs test, prédictions et
   RMSE (19,70) sont identiques pour tous les facteurs k, alors que le score passe de 852,9 (k = 1) à 0,13
   (k = 1 000). « Optimiser » ces constantes reviendrait à choisir la note qui flatte le plus, pas un meilleur modèle.
2. **Mettre la pénalité dans l'apprentissage fonctionne, mais rien n'est gratuit.** Avec la perte quadratique
   asymétrique, quand α passe de 1 à 3, les surestimés OOF > 20 baissent de 2 386 à 1 590 (−33 %) pour
   XGBoost et de 2 316 à 1 675 (−28 %) pour LightGBM ; en échange, le RMSE CV de XGBoost monte de 16,86 à
   17,68, les fausses alertes de 266 à 435 (+64 %) et le biais passe de −0,2 à −4,2 cycles (le modèle
   sous-estime de plus en plus). Le score NASA CV baisse jusqu'à α = 2 (25 514) puis plus du tout (α = 3 :
   25 527) : au-delà, les sous-estimations, elles aussi pénalisées par la note, compensent le gain.
3. **α = 1 redonne la Phase 3** (RMSE CV 16,86 contre 16,87) : contrôle que la perte sur mesure est bien branchée.
4. **Perte de forme NASA : attention à la convergence.** Avec 200 arbres, la perte NASA (13, 10) donne le
   meilleur compromis de la grille (RMSE 16,85, comme la Phase 3 ; score NASA 22 235, soit −25 % ;
   surestimés 1 806, soit −24 %). Mais le diagnostic montre que ce modèle **n'a pas convergé** : avec 1 000
   puis 2 000 arbres (chiffres presque identiques entre les deux, donc convergé), il devient **moins bon que
   la Phase 3 sur les deux critères** (RMSE 17,51, score NASA 31 704). Le gain à 200 arbres vient donc de la
   combinaison « perte NASA + apprentissage arrêté tôt » (biais −2,7), pas de la perte seule ; il se paie
   aussi en fausses alertes (310 contre 259) et en régularité entre plis (écart-type 1,78 contre 1,25).
   Avec h_sur = 5, 200 arbres sont très loin de la convergence (RMSE 20,06, biais −7,7).
5. **Conséquence** : la pénalité, le nombre d'arbres et le pas doivent être réglés **ensemble** — c'est
   exactement ce que fait la recherche Optuna de la section D.

## D. Recherche systématique avec Optuna

### D.1 Principe

**Optuna** essaie automatiquement des combinaisons de réglages et garde la trace de leurs résultats.
Chaque « essai » = une configuration évaluée en validation croisée sur **les mêmes 5 plis**. L'algorithme
**TPE** tire d'abord 10 essais au hasard, puis propose des réglages qui ressemblent aux meilleurs essais
déjà vus (graine fixée à 42 : mêmes essais à chaque relance).

**Deux objectifs à la fois** (multi-objectif) : minimiser le RMSE CV moyen **et** le score NASA CV moyen.
Optuna ne mélange pas les deux en une note unique ; il garde le **front de Pareto** : les essais qu'aucun
autre essai ne bat sur les deux critères à la fois. Exemple : A (RMSE 16,0 ; NASA 30 000) et B (17,0 ;
25 000) sont tous les deux sur le front (chacun gagne sur un critère) ; C (17,0 ; 31 000) est dominé par A.

Trois études (espaces de recherche dans `src/optimisation.py`) :

| Étude | Modèle et perte d'apprentissage | Paramètres de perte cherchés | Essais |
|---|---|---|---|
| `xgb_quadratique` | XGBoost, quadratique asymétrique | α ∈ [1, 4] | 60 |
| `xgb_nasa` | XGBoost, forme NASA (l'idée du coach, au bon endroit) | h_sur ∈ [3, 15], h_sous ∈ [8, 30] | 60 |
| `lgbm_quadratique` | LightGBM, quadratique asymétrique | α ∈ [1, 4] | 40 |

Hyperparamètres XGBoost : n_estimators [100, 800] (jusqu'à 1 200 pour la perte NASA, qui converge plus
lentement), learning_rate [0,01 ; 0,2] (jusqu'à 0,3 pour NASA), max_depth [3, 8], min_child_weight [1, 20],
subsample [0,6 ; 1], colsample_bytree [0,5 ; 1], reg_lambda [0,1 ; 10]. LightGBM : n_estimators [100, 600],
learning_rate [0,01 ; 0,2], num_leaves [15, 127], min_child_samples [5, 100], subsample [0,6 ; 1] (re-tiré à
chaque arbre), colsample_bytree [0,5 ; 1], reg_lambda [0,001 ; 10]. LightGBM n'a que 40 essais : en mode
déterministe, un essai coûte environ 2 à 4 fois plus cher qu'un essai XGBoost (mesuré avant le lancement).

In [ ]:
if RELANCER_OPTUNA:
    # calcul lourd : les 3 etudes, l'une apres l'autre (meme code que scripts/lancer_optuna.py)
    for nom_etude in NOMS_ETUDES:
        etude, tableau_etude = lancer_etude(nom_etude, X, y, groups, plis, n_essais=N_ESSAIS_PAR_DEFAUT[nom_etude])
        tableau_etude.to_csv(chemin_csv_etude(nom_etude), index=False)

essais = charger_essais(NOMS_ETUDES)  # relit docs/resultats/04_optuna_<etude>.csv
print(f"{len(essais)} essais relus.")

lignes = []
for nom_etude in NOMS_ETUDES:
    essais_etude = essais[essais["etude"] == nom_etude]
    ligne = {}
    ligne["etude"] = nom_etude
    ligne["n_essais"] = len(essais_etude)
    ligne["rmse_cv_min"] = essais_etude["rmse_cv_moyen"].min()
    ligne["nasa_cv_min"] = essais_etude["nasa_cv_moyen"].min()
    ligne["surestimes_oof_min"] = essais_etude["surestimes_oof"].min()
    ligne["n_pareto_dans_l_etude"] = int(essais_etude["pareto_etude"].sum())
    ligne["duree_moyenne_essai_s"] = essais_etude["duree_essai_s"].mean()
    ligne["duree_etude_min"] = essais_etude["duree_etude_totale_s"].iloc[0] / 60.0
    lignes.append(ligne)
pd.DataFrame(lignes).set_index("etude").round(2)

### D.2 Front de Pareto, toutes études confondues

On recalcule le front sur l'ensemble des essais des trois études (`calculer_front_pareto`) : un essai
Pareto-optimal dans son étude peut être dominé par un essai d'une autre étude.

In [ ]:
essais["pareto_global"] = calculer_front_pareto(essais, "rmse_cv_moyen", "nasa_cv_moyen")
front_global = essais[essais["pareto_global"]].sort_values("rmse_cv_moyen")

colonnes_front = ["etude", "numero", "rmse_cv_moyen", "rmse_cv_ecart_type", "nasa_cv_moyen", "surestimes_oof",
                  "alertes_manquees_oof", "fausses_alertes_oof", "biais_oof", "duree_essai_s"]
print(f"{len(front_global)} essais sur le front de Pareto global.")
front_global[colonnes_front].round(2)

### D.3 Application de la règle de choix

La règle a été écrite **avant** de lancer les études (constante `TEXTE_REGLE_DE_CHOIX` et fonction
`appliquer_regle_de_choix`, testée sur un exemple calculé à la main dans `tests/test_optimisation.py`) :

1. front de Pareto (RMSE CV moyen, score NASA CV moyen) de tous les essais des trois études ;
2. on garde ceux dont le RMSE CV ≤ 1,05 × le plus petit RMSE CV du front (on accepte de perdre au plus 5 %
   de précision) ;
3. parmi eux, le plus petit score NASA CV ; en cas d'égalité, le moins de surestimations OOF > 20.

On garde aussi **« le meilleur RMSE CV »** comme second candidat. Le jeu de test n'intervient pas.

In [ ]:
print(TEXTE_REGLE_DE_CHOIX)
choix = appliquer_regle_de_choix(essais, tolerance=TOLERANCE_RMSE)

print(f"\nPlus petit RMSE CV du front : {choix['rmse_min_front']:.3f} -> seuil 1,05 x = {choix['seuil_rmse']:.3f}")
print(f"{len(choix['admissibles'])} essais du front sous le seuil (sur {len(choix['front'])} essais du front).")
choix["admissibles"][colonnes_front].round(2)

In [ ]:
essai_retenu = choix["retenu"]
essai_meilleur_rmse = choix["meilleur_rmse"]

config_retenue = config_depuis_ligne(essai_retenu)  # famille, perte, parametres_perte, hyperparametres
config_meilleur_rmse = config_depuis_ligne(essai_meilleur_rmse)
NOM_RETENU = f"Optuna retenu ({essai_retenu['etude']} n°{int(essai_retenu['numero'])})"
NOM_MEILLEUR_RMSE = f"Optuna meilleur RMSE ({essai_meilleur_rmse['etude']} n°{int(essai_meilleur_rmse['numero'])})"

for nom_candidat, essai, config in [(NOM_RETENU, essai_retenu, config_retenue),
                                    (NOM_MEILLEUR_RMSE, essai_meilleur_rmse, config_meilleur_rmse)]:
    print(f"=== {nom_candidat} ===")
    print(f"RMSE CV = {essai['rmse_cv_moyen']:.3f} (ecart-type entre plis {essai['rmse_cv_ecart_type']:.2f}), "
          f"NASA CV = {essai['nasa_cv_moyen']:.1f}, surestimes OOF > 20 = {int(essai['surestimes_oof'])}, "
          f"biais OOF = {essai['biais_oof']:+.2f}")
    print(json.dumps(config, indent=2))
print(f"Les deux candidats sont le meme essai : {choix['meme_essai']}")

### D.4 Figure du front de Pareto

Chaque point est un essai (couleur = étude). Les essais du front global sont cerclés de noir et reliés ; la
ligne verticale pointillée est le seuil de la règle (1,05 × le meilleur RMSE du front) : le retenu est le
point du front le plus bas (score NASA) à gauche de cette ligne. Les modèles de base de la section B
servent de repères. Le cadre est resserré sur la zone utile ; le nombre d'essais hors cadre est indiqué
dans le titre.

In [ ]:
couleurs_etudes = {"xgb_quadratique": COULEUR_1, "xgb_nasa": COULEUR_3, "lgbm_quadratique": COULEUR_2}

# cadre : un peu de marge autour du front et des reperes ; les essais tres mauvais sortent du cadre
limite_x_haute = max(essais["rmse_cv_moyen"].quantile(0.85), front_global["rmse_cv_moyen"].max() + 0.3)
limite_y_haute = max(essais["nasa_cv_moyen"].quantile(0.85), resultats_cv_base["LightGBM"]["nasa_moyen"]) * 1.08
limite_x_basse = essais["rmse_cv_moyen"].min() - 0.2
limite_y_basse = essais["nasa_cv_moyen"].min() * 0.95
hors_cadre = (essais["rmse_cv_moyen"] > limite_x_haute) | (essais["nasa_cv_moyen"] > limite_y_haute)

fig, ax = plt.subplots(figsize=(11, 7))
for nom_etude in NOMS_ETUDES:
    essais_etude = essais[essais["etude"] == nom_etude]
    ax.scatter(essais_etude["rmse_cv_moyen"], essais_etude["nasa_cv_moyen"], s=30, alpha=0.6,
               color=couleurs_etudes[nom_etude], label=f"{nom_etude} ({len(essais_etude)} essais)")

# front global : points cercles de noir, relies en escalier (dans l'ordre du RMSE)
ax.step(front_global["rmse_cv_moyen"], front_global["nasa_cv_moyen"], where="post", color="black", linewidth=1)
ax.scatter(front_global["rmse_cv_moyen"], front_global["nasa_cv_moyen"], s=70, facecolors="none",
           edgecolors="black", linewidths=1.5, label="front de Pareto global")

# seuil de la regle de choix, et les deux candidats
ax.axvline(choix["seuil_rmse"], color=COULEUR_TEXTE, linestyle=":", label="seuil de la règle (1,05 × meilleur RMSE)")
candidats_figure = [(essai_retenu, "retenu", (-60, -45)), (essai_meilleur_rmse, "meilleur RMSE", (20, 45))]
for essai, texte, decalage in candidats_figure:  # etiquette en gras + fleche vers le point exact
    ax.annotate(f"{texte} ({essai['etude']} n°{int(essai['numero'])})", (essai["rmse_cv_moyen"], essai["nasa_cv_moyen"]),
                textcoords="offset points", xytext=decalage, ha="center", fontsize=9, fontweight="bold",
                arrowprops={"arrowstyle": "->", "color": "black"})

# reperes : modeles de base de la section B (decalages choisis pour que les etiquettes ne se chevauchent pas)
reperes = [("XGBoost Phase 3", "*", (-10, -14), "right"), ("Random Forest", "^", (10, -4), "left"),
           ("LightGBM", "P", (10, 4), "left")]
for nom_modele, marqueur, decalage, alignement in reperes:
    resultats = resultats_cv_base[nom_modele]
    ax.scatter(resultats["rmse_moyen"], resultats["nasa_moyen"], marker=marqueur, s=180, color="black", zorder=4)
    ax.annotate(nom_modele + " (base)", (resultats["rmse_moyen"], resultats["nasa_moyen"]),
                textcoords="offset points", xytext=decalage, ha=alignement, fontsize=9)

ax.set_xlim(limite_x_basse, limite_x_haute)
ax.set_ylim(limite_y_basse, limite_y_haute)
ax.set_xlabel("RMSE CV moyen (cycles) — plus à gauche = plus précis")
ax.set_ylabel("score NASA CV moyen (13/10) — plus bas = plus prudent")
ax.set_title(f"Optuna multi-objectif : {len(essais)} essais, 3 études, mêmes 5 plis "
             f"({int(hors_cadre.sum())} essais hors cadre)")
ax.legend(fontsize=9, loc="upper right")
plt.tight_layout()
plt.savefig(os.path.join(DOSSIER_FIGURES, "04_pareto_optuna.png"), dpi=150)
plt.show()

### D.5 Quels réglages comptent ? (corrélations de rang, indicatif)

Pour chaque étude, la corrélation de Spearman entre chaque réglage et chacun des deux objectifs. Un ρ
positif avec le score NASA veut dire « quand ce réglage augmente, le score NASA augmente (se dégrade) ».
C'est **indicatif** : TPE ne tire pas les réglages indépendamment ni uniformément (il se concentre sur les
bonnes zones), et 40 à 60 essais ne permettent pas de séparer finement des effets qui interagissent.

In [ ]:
lignes = []
for nom_etude in NOMS_ETUDES:
    essais_etude = essais[essais["etude"] == nom_etude]
    for colonne in essais_etude.columns:
        if not colonne.startswith("param_"):
            continue  # seulement les reglages tires par Optuna
        if essais_etude[colonne].isna().all():
            continue  # reglage d'une autre etude (ex. alpha n'existe pas dans xgb_nasa)
        valeurs = essais_etude[colonne].to_numpy(dtype=float)
        rho_rmse = stats.spearmanr(valeurs, essais_etude["rmse_cv_moyen"]).statistic
        rho_nasa = stats.spearmanr(valeurs, essais_etude["nasa_cv_moyen"]).statistic
        ligne = {}
        ligne["etude"] = nom_etude
        ligne["reglage"] = colonne.replace("param_", "")
        ligne["rho_avec_rmse_cv"] = rho_rmse
        ligne["rho_avec_nasa_cv"] = rho_nasa
        lignes.append(ligne)
tableau_spearman = pd.DataFrame(lignes).set_index(["etude", "reglage"])
tableau_spearman.round(2)

### D.6 Contrôle : la configuration retenue se reconstruit à l'identique

On reconstruit chaque candidat **à partir de sa configuration seule** (comme le fera la démo, via
`construire_fabrique_depuis_config`), on relance la validation croisée sur les mêmes plis, et on vérifie
qu'on retrouve exactement les chiffres du CSV Optuna. Cela prouve que le fichier de configuration suffit à
reproduire le modèle.

In [ ]:
candidats_optuna = {}  # nom -> configuration
candidats_optuna[NOM_RETENU] = config_retenue
if not choix["meme_essai"]:
    candidats_optuna[NOM_MEILLEUR_RMSE] = config_meilleur_rmse

lignes_csv = {}
lignes_csv[NOM_RETENU] = essai_retenu
lignes_csv[NOM_MEILLEUR_RMSE] = essai_meilleur_rmse

resultats_cv_optuna = {}
for nom_candidat in candidats_optuna:
    fabrique = construire_fabrique_depuis_config(candidats_optuna[nom_candidat])
    resultats_cv_optuna[nom_candidat] = evaluer_en_validation_croisee(
        fabrique, X, y, groups, plis, seuil_surestimation=SEUIL_SURESTIMATION, seuil_alerte=SEUIL_ALERTE
    )
    rmse_recalcule = resultats_cv_optuna[nom_candidat]["rmse_moyen"]
    rmse_csv = lignes_csv[nom_candidat]["rmse_cv_moyen"]
    nasa_recalcule = resultats_cv_optuna[nom_candidat]["nasa_moyen"]
    nasa_csv = lignes_csv[nom_candidat]["nasa_cv_moyen"]
    print(f"{nom_candidat} : RMSE CV recalcule {rmse_recalcule:.6f} vs CSV {rmse_csv:.6f} ; "
          f"NASA CV recalcule {nasa_recalcule:.3f} vs CSV {nasa_csv:.3f}")
    assert abs(rmse_recalcule - rmse_csv) < 1e-6, "la configuration ne reproduit pas le RMSE CV de l'essai"
    assert abs(nasa_recalcule - nasa_csv) < 1e-3, "la configuration ne reproduit pas le score NASA CV de l'essai"

**Les gains en validation croisée sont-ils significatifs ?** Même principe qu'en section B : des valeurs
**par moteur** (100 moteurs), candidat contre XGBoost Phase 3, test t apparié et Wilcoxon. Trois métriques
par moteur : RMSE, pénalité NASA (somme des pénalités des cycles du moteur) et nombre de cycles surestimés
de plus de 20. Différence négative = le candidat fait mieux.

Attention : les candidats ont été *choisis* parmi 160 essais sur ces mêmes plis, ce qui rend ces tests
**optimistes** (le meilleur de 160 tirages profite aussi d'un peu de chance). Une p-valeur un peu sous 0,05
n'est donc pas une preuve solide ici ; seule une p-valeur très petite résiste à cet effet de sélection.
D'où l'intérêt de l'évaluation finale sur le test, jamais vu pendant le choix.

In [ ]:
metriques_moteur_phase3_cv = metriques_par_moteur(y, resultats_cv_base["XGBoost Phase 3"]["predictions_oof"], groups)

lignes = []
for nom_candidat in resultats_cv_optuna:
    metriques_moteur_candidat = metriques_par_moteur(y, resultats_cv_optuna[nom_candidat]["predictions_oof"], groups)
    for metrique in ["rmse", "nasa", "surestimes"]:
        valeurs_candidat = metriques_moteur_candidat[metrique].to_numpy(dtype=float)
        valeurs_phase3 = metriques_moteur_phase3_cv[metrique].to_numpy(dtype=float)
        comparaison = comparer_apparie(valeurs_candidat, valeurs_phase3)
        ligne = {}
        ligne["candidat (vs XGBoost Phase 3)"] = nom_candidat
        ligne["metrique_par_moteur"] = metrique
        ligne["moyenne_phase3"] = float(np.mean(valeurs_phase3))
        ligne["moyenne_candidat"] = float(np.mean(valeurs_candidat))
        ligne["difference_moyenne"] = comparaison["difference_moyenne"]
        ligne["moteurs_ou_le_candidat_est_meilleur"] = int(np.sum(valeurs_candidat < valeurs_phase3))
        ligne["moteurs_ou_le_candidat_est_moins_bon"] = int(np.sum(valeurs_candidat > valeurs_phase3))
        ligne["p_valeur_t"] = comparaison["p_valeur_t"]
        ligne["p_valeur_wilcoxon"] = comparaison["p_valeur_wilcoxon"]
        lignes.append(ligne)
tableau_tests_cv_optuna = pd.DataFrame(lignes).set_index(["candidat (vs XGBoost Phase 3)", "metrique_par_moteur"])
tableau_tests_cv_optuna.round(4)

### D.7 Conclusion de la section D

- **Coût** : 160 essais (60 + 60 + 40) en 17,2 minutes, les trois études tournant en parallèle
  (`scripts/lancer_optuna.py --parallele`, 3 × 4 cœurs). Un essai LightGBM coûte en moyenne 26 s, contre 7 s
  et 12 s pour les études XGBoost : avec 40 essais seulement, la comparaison entre familles est un peu
  défavorable à LightGBM.
- **Le front de Pareto global (12 essais) ne contient que des XGBoost** (6 de `xgb_nasa`, 6 de
  `xgb_quadratique`) ; le meilleur essai LightGBM a un RMSE CV de 16,85.
- **Optuna trouve des réglages meilleurs que XGBoost Phase 3 sur les deux critères à la fois** : tout le
  front a un score NASA CV plus bas que la Phase 3, et 7 de ses 12 points ont aussi un RMSE CV plus bas.
  Le meilleur RMSE (`xgb_nasa` n°42 : perte NASA presque symétrique, h_sous ≈ h_sur ≈ 13,6 ; 573 arbres de
  profondeur 4, pas 0,023) atteint un RMSE CV de 16,48 (−0,39) et un score NASA CV de 24 586 (−17 %).
- **Corrélations (indicatif)** : la pénalité pilote bien le compromis (α : ρ = +0,64 avec le RMSE et −0,38
  avec le score NASA pour XGBoost, +0,70 / −0,58 pour LightGBM ; h_sur : ρ = −0,59 avec le RMSE), et des
  arbres plus complexes dégradent le score NASA (max_depth et num_leaves : ρ = +0,82).
- **La règle de choix désigne `xgb_quadratique` n°17** : perte quadratique asymétrique avec α = 2,26 ;
  200 arbres de profondeur 3, pas 0,021, min_child_weight 8,7, subsample 0,82, colsample_bytree 0,77,
  reg_lambda 1,11. Seuil : 1,05 × 16,48 = 17,31 ; 11 des 12 essais du front sont admissibles, et le retenu
  (RMSE CV 17,28) est le plus prudent d'entre eux. Par construction, la règle prend le point du front le plus
  prudent **dans le budget de 5 %** : il se trouve donc au bord de ce budget, et le choix dépend de la valeur
  1,05, fixée avant les résultats et non retouchée depuis.
- **Ce que le retenu change en validation croisée, contre XGBoost Phase 3** (tests appariés sur 100 moteurs) :
  - surestimés > 20 cycles : 1 517 lignes OOF contre 2 384 (−36 %) ; 58 moteurs améliorés, 4 dégradés ;
    p < 0,0001 (t et Wilcoxon). **Net**, et cohérent avec la grille de la section C (tout α > 1 réduit les
    surestimations) ;
  - score NASA CV : 21 075 contre 29 783 (−29 %) ; par moteur, p = 0,007 (t) et 0,028 (Wilcoxon).
    **Significatif, mais modérément** : avec l'effet de sélection parmi 160 essais, ce n'est pas une preuve
    solide à lui seul ;
  - RMSE : 17,28 contre 16,87 en moyenne des plis (+2,4 %) ; par moteur, aucune différence détectable
    (p = 0,67 / 0,61) ;
  - alertes manquées : 492 contre 552 (−11 %) ;
  - **en contrepartie** : fausses alertes 381 contre 259 (+47 %), biais −4,3 cycles (sous-estimation
    moyenne), écart-type entre plis 2,10 contre 1,25 (moins régulier d'un groupe de moteurs à l'autre).
- **Le meilleur RMSE** améliore aussi les deux critères par moteur (RMSE : p = 0,015 / 0,039 ; NASA :
  p = 0,0007 / < 0,0001), mais réduit peu les surestimations (2 261 contre 2 384).
- La configuration se reconstruit à l'identique à partir du seul JSON (contrôle D.6) : la démo utilisera
  exactement ce modèle.

## E. Évaluation finale sur le jeu de test officiel (une seule fois)

Le choix est fait (section D) : on peut maintenant regarder le test. Cinq modèles, chacun entraîné sur les
**100 moteurs** d'entraînement puis évalué sur le dernier cycle de chaque moteur test, comparé à la RUL
**brute** de `RUL_FD001.txt` (protocole Phase 3 ; prédictions clippées à 0). On ajoute le RMSE contre la RUL
vraie **plafonnée** à 125 (le modèle ne peut pas prédire au-delà de 125 : ce chiffre mesure l'erreur sur ce
qu'il pouvait réellement apprendre). Rien de ce qui suit ne modifie le choix.

**Contrôle de reproduction** : XGBoost Phase 3 doit redonner RMSE 19,70, score NASA 852,9 et 9 surestimés.

In [ ]:
configs_finales = {}
configs_finales["XGBoost Phase 3"] = CONFIG_XGB_PHASE3
configs_finales["Random Forest"] = CONFIG_RANDOM_FOREST
configs_finales["LightGBM"] = CONFIG_LIGHTGBM_BASE
if not choix["meme_essai"]:
    configs_finales[NOM_MEILLEUR_RMSE] = config_meilleur_rmse
configs_finales[NOM_RETENU] = config_retenue

moteurs_test_ordre = contexte_test["moteurs_test"]  # numeros des moteurs test, meme ordre que y_test_vraie
resultats_test = {}  # nom -> dictionnaire de evaluer_sur_test
for nom_modele in configs_finales:
    modele_final = construire_fabrique_depuis_config(configs_finales[nom_modele])()  # modele neuf
    modele_final.fit(X, y)  # entraine sur les 100 moteurs d'entrainement
    resultats_test[nom_modele] = evaluer_sur_test(modele_final, X_test_dernier, y_test_vraie, plafond=PLAFOND_RUL,
                                                  seuil_surestimation=SEUIL_SURESTIMATION, seuil_alerte=SEUIL_ALERTE)

# controle de reproduction du test de la Phase 3
assert round(resultats_test["XGBoost Phase 3"]["rmse"], 2) == 19.70
assert round(resultats_test["XGBoost Phase 3"]["score_nasa"], 1) == 852.9
assert resultats_test["XGBoost Phase 3"]["nb_surestimes"] == 9

lignes = []
for nom_modele in resultats_test:
    lignes.append(resumer_test(nom_modele, resultats_test[nom_modele]))
tableau_test = pd.DataFrame(lignes).set_index("modele")
tableau_test.round(2)

### E.1 Test apparié sur les 100 moteurs test : retenu vs Phase 3

Erreur absolue de chaque moteur test, modèle retenu contre XGBoost Phase 3 (mêmes 100 moteurs). Différence
moyenne négative = le retenu se trompe moins. Avec 100 points seulement, il faut un écart net pour conclure.

In [ ]:
erreurs_absolues_phase3 = np.abs(resultats_test["XGBoost Phase 3"]["predictions"] - y_test_vraie)

lignes = []
for nom_modele in resultats_test:
    if nom_modele == "XGBoost Phase 3":
        continue
    erreurs_absolues_modele = np.abs(resultats_test[nom_modele]["predictions"] - y_test_vraie)
    comparaison = comparer_apparie(erreurs_absolues_modele, erreurs_absolues_phase3)
    ligne = {}
    ligne["modele (vs XGBoost Phase 3)"] = nom_modele
    ligne["n_moteurs_test"] = comparaison["n"]
    ligne["difference_moyenne_erreur_absolue"] = comparaison["difference_moyenne"]
    ligne["moteurs_ou_le_modele_est_meilleur"] = int(np.sum(erreurs_absolues_modele < erreurs_absolues_phase3))
    ligne["p_valeur_t"] = comparaison["p_valeur_t"]
    ligne["p_valeur_wilcoxon"] = comparaison["p_valeur_wilcoxon"]
    lignes.append(ligne)
tableau_tests_test = pd.DataFrame(lignes).set_index("modele (vs XGBoost Phase 3)")
tableau_tests_test.round(4)

**Et le score NASA du test ?** Le score NASA est une somme de pénalités exponentielles : une seule grosse
surestimation peut en faire l'essentiel. On regarde donc (1) la pénalité NASA de chaque moteur test,
comparée moteur par moteur (test apparié, retenu et meilleur RMSE contre Phase 3) ; (2) les moteurs qui
pèsent le plus dans le score de la Phase 3 ; (3) le score total **sans** le moteur le plus pénalisé.

In [ ]:
penalites_test = {}  # nom -> penalite NASA de chacun des 100 moteurs test (un point par moteur)
for nom_modele in resultats_test:
    penalites_test[nom_modele] = metriques_par_moteur(y_test_vraie, resultats_test[nom_modele]["predictions"],
                                                      moteurs_test_ordre)["nasa"].to_numpy()

lignes = []
for nom_modele in resultats_test:
    if nom_modele == "XGBoost Phase 3":
        continue
    comparaison = comparer_apparie(penalites_test[nom_modele], penalites_test["XGBoost Phase 3"])
    ligne = {}
    ligne["modele (vs XGBoost Phase 3)"] = nom_modele
    ligne["difference_moyenne_penalite_nasa"] = comparaison["difference_moyenne"]
    ligne["moteurs_ou_le_modele_est_meilleur"] = int(np.sum(penalites_test[nom_modele] < penalites_test["XGBoost Phase 3"]))
    ligne["p_valeur_t"] = comparaison["p_valeur_t"]
    ligne["p_valeur_wilcoxon"] = comparaison["p_valeur_wilcoxon"]
    lignes.append(ligne)
print("Penalite NASA par moteur test, tests apparies contre XGBoost Phase 3 :")
print(pd.DataFrame(lignes).set_index("modele (vs XGBoost Phase 3)").round(4).to_string())

# les 5 moteurs qui pesent le plus dans le score NASA de la Phase 3, et leur penalite avec chaque modele
ordre = np.argsort(-penalites_test["XGBoost Phase 3"])  # du plus penalise au moins penalise
lignes = []
for position in ordre[:5]:
    ligne = {}
    ligne["unit_number"] = moteurs_test_ordre[position]
    ligne["rul_vraie"] = y_test_vraie[position]
    for nom_modele in resultats_test:
        ligne[nom_modele] = penalites_test[nom_modele][position]
    lignes.append(ligne)
print("\nLes 5 moteurs les plus penalises par la Phase 3 (penalite NASA de chaque modele) :")
print(pd.DataFrame(lignes).set_index("unit_number").round(1).to_string())

# score total avec et sans le moteur le plus penalise par la Phase 3
position_pire = ordre[0]
print(f"\nScore NASA test total, avec et sans le moteur {moteurs_test_ordre[position_pire]} :")
for nom_modele in resultats_test:
    total = float(np.sum(penalites_test[nom_modele]))
    sans_le_pire = total - penalites_test[nom_modele][position_pire]
    print(f"  {nom_modele:40s} : {total:7.1f} avec, {sans_le_pire:7.1f} sans")

### E.2 Robustesse à la graine du modèle

Un bon chiffre peut venir d'une graine chanceuse. On réentraîne le modèle retenu et XGBoost Phase 3 avec
5 graines différentes (`random_state` = 0 à 4) et on regarde la moyenne et l'écart-type des métriques test.
La graine ne joue que si le modèle tire au hasard des lignes ou des colonnes (`subsample` ou
`colsample_bytree` < 1) : XGBoost Phase 3 n'en tire aucune, il doit donc être **identique** pour les 5 graines.

In [ ]:
def config_avec_graine(config, graine):
    """
    Renvoie une copie de la configuration avec une autre graine de modele (random_state).

    Parameters
    ----------
    config : dict
        La configuration d'origine (non modifiee).
    graine : int
        La nouvelle valeur de random_state.

    Returns
    -------
    dict
        Une copie independante, avec hyperparametres["random_state"] = graine.
    """
    copie = copy.deepcopy(config)
    copie["hyperparametres"]["random_state"] = graine
    return copie


lignes = []
for nom_modele in ["XGBoost Phase 3", NOM_RETENU]:
    for graine in range(5):  # graines 0 a 4
        config_graine = config_avec_graine(configs_finales[nom_modele], graine)
        modele_graine = construire_fabrique_depuis_config(config_graine)()
        modele_graine.fit(X, y)
        resultat = evaluer_sur_test(modele_graine, X_test_dernier, y_test_vraie, plafond=PLAFOND_RUL,
                                    seuil_surestimation=SEUIL_SURESTIMATION, seuil_alerte=SEUIL_ALERTE)
        ligne = resumer_test(nom_modele, resultat)
        ligne["graine"] = graine
        lignes.append(ligne)
tableau_graines = pd.DataFrame(lignes)

colonnes_graines = ["rmse_test", "rmse_test_rul_plafonnee", "mae_test", "nasa_test", "surestimes_test",
                    "alertes_manquees_test", "fausses_alertes_test"]
resume_graines = tableau_graines.groupby("modele")[colonnes_graines].agg(["mean", "std"])
resume_graines.round(2)

In [ ]:
# le sous-echantillonnage du modele retenu (cause possible de sensibilite a la graine)
hyperparametres_retenu = config_retenue["hyperparametres"]
print("Modele retenu : subsample =", hyperparametres_retenu.get("subsample"),
      "| colsample_bytree =", hyperparametres_retenu.get("colsample_bytree"))
ecart_type_rmse_phase3 = tableau_graines[tableau_graines["modele"] == "XGBoost Phase 3"]["rmse_test"].std()
print(f"XGBoost Phase 3 : ecart-type du RMSE test entre les 5 graines = {ecart_type_rmse_phase3:.6f}")

### E.3 Figure : prédictions vs vraies RUL sur le test (Phase 3 et retenu côte à côte)

Même lecture qu'en Phase 3 : la diagonale est la prédiction parfaite, les points au-dessus sont des
surestimations. Les moteurs surestimés de plus de 20 cycles sont mis en évidence (orange, triangles).

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 7), sharex=True, sharey=True)
limite = max(y_test_vraie.max(), resultats_test[NOM_RETENU]["predictions"].max(),
             resultats_test["XGBoost Phase 3"]["predictions"].max()) + 5

for ax, nom_modele in zip(axes, ["XGBoost Phase 3", NOM_RETENU]):
    predictions = resultats_test[nom_modele]["predictions"]
    ecarts = predictions - y_test_vraie
    surestime = ecarts > SEUIL_SURESTIMATION  # moteurs surestimes de plus de 20 cycles

    ax.scatter(y_test_vraie[~surestime], predictions[~surestime], alpha=0.6, color=COULEUR_1, label="moteurs test")
    ax.scatter(y_test_vraie[surestime], predictions[surestime], alpha=0.9, color=COULEUR_2, marker="^", s=60,
               label=f"surestimés > {SEUIL_SURESTIMATION} cycles ({int(surestime.sum())})")
    ax.plot([0, limite], [0, limite], linestyle="--", color="black", label="prédiction = vérité")
    ax.plot([0, limite], [SEUIL_SURESTIMATION, limite + SEUIL_SURESTIMATION], linestyle=":", color=COULEUR_TEXTE,
            label=f"vérité + {SEUIL_SURESTIMATION}")
    resultats = resultats_test[nom_modele]
    ax.set_title(f"{nom_modele}\nRMSE={resultats['rmse']:.2f}, score NASA={resultats['score_nasa']:.1f}, "
                 f"surestimés>20={resultats['nb_surestimes']}/100", fontsize=10)
    ax.set_xlabel("RUL vraie (RUL_FD001.txt)")
    ax.set_xlim(0, limite)
    ax.set_ylim(0, limite)
    ax.legend(fontsize=8, loc="upper left")
axes[0].set_ylabel("RUL prédite (modèle final)")
fig.suptitle("Test officiel FD001 — évalué une seule fois, après le choix")
plt.tight_layout()
plt.savefig(os.path.join(DOSSIER_FIGURES, "04_predictions_vs_vraies_rul_test_retenu.png"), dpi=150)
plt.show()

### E.4 Conclusion de la section E

- **Contrôle** : XGBoost Phase 3 redonne exactement les chiffres publiés (RMSE 19,70, score NASA 852,9,
  9 surestimés).
- **Retenu contre Phase 3 sur le test** : RMSE 19,73 contre 19,70 ; MAE 14,32 contre 14,36 ; surestimés > 20 :
  9 contre 9 ; alertes manquées 7 contre 7 ; fausses alertes 1 contre 1 ; RMSE contre la RUL plafonnée 18,31
  contre 18,53 ; score NASA 675,4 contre 852,9 (−21 %).
- **Mais ce gain de score NASA tient à un seul moteur** : le moteur 37 (RUL vraie 21) est surestimé de
  54 cycles par la Phase 3 (pénalité 223,6, plus du quart de son score) et de 34 cycles par le retenu
  (pénalité 29,3). Sans ce moteur, les deux modèles sont au même niveau (646,1 pour le retenu, 629,3 pour la
  Phase 3). Le test apparié sur la pénalité NASA par moteur ne détecte aucune différence (p = 0,38 / 0,61),
  pas plus que sur l'erreur absolue (p = 0,94 / 0,99).
- **Aucun modèle n'est significativement différent de XGBoost Phase 3 sur le test** (erreur absolue par
  moteur : toutes les p-valeurs ≥ 0,08). Le meilleur RMSE d'Optuna a les meilleurs chiffres bruts (RMSE 18,69,
  score NASA 646,9), sans écart significatif non plus (p = 0,11 / 0,33).
- **Robustesse à la graine** : sur 5 graines, le retenu donne un RMSE test de 19,65 ± 0,05 et un score NASA de
  666 ± 6, toujours 9 surestimés : le résultat ne dépend pas d'une graine chanceuse. XGBoost Phase 3 est
  strictement identique pour les 5 graines (aucun sous-échantillonnage : la graine n'est jamais utilisée).

**Lecture honnête** : le test officiel (100 points) **ne confirme pas et ne contredit pas** le gain de
prudence mesuré en validation croisée (20 631 lignes, 100 moteurs) : il n'a pas la puissance statistique
pour trancher. L'indicateur métier le plus critique, les alertes manquées, vaut 7 pour les 5 modèles. Ce qui
est établi l'est en validation croisée (section D) ; le test montre que le retenu ne dégrade pas la
précision et qu'il réduit la pire surestimation (54 → 34 cycles).

## F. Les moteurs surestimés de plus de 20 cycles (action demandée par le coach)

On liste, pour XGBoost Phase 3 et pour le modèle retenu, les moteurs test surestimés de plus de 20 cycles,
avec le nombre de cycles **observés** dans `test_FD001.txt` (la longueur de la trajectoire tronquée dont
dispose le modèle). Cette analyse vient **après** le choix : elle sert à comprendre les erreurs, pas à
re-choisir un modèle.

In [ ]:
# nombre de cycles observes pour chaque moteur test (trajectoire tronquee), dans l'ordre des moteurs
nb_cycles_par_moteur = contexte_test["test"].groupby("unit_number").size()
moteurs_test = contexte_test["moteurs_test"]  # numeros de moteur, meme ordre que y_test_vraie
nb_cycles_observes = nb_cycles_par_moteur.loc[moteurs_test].to_numpy()

tableaux_surestimes = {}
for nom_modele in ["XGBoost Phase 3", NOM_RETENU]:
    predictions = resultats_test[nom_modele]["predictions"]
    tableau_moteurs = pd.DataFrame({
        "unit_number": moteurs_test,
        "rul_vraie": y_test_vraie,
        "rul_predite": predictions,
        "ecart": predictions - y_test_vraie,  # positif = surestimation
        "nb_cycles_observes": nb_cycles_observes,
    })
    tableau_moteurs["surestime_20"] = tableau_moteurs["ecart"] > SEUIL_SURESTIMATION
    tableaux_surestimes[nom_modele] = tableau_moteurs
    print(f"=== {nom_modele} : {int(tableau_moteurs['surestime_20'].sum())} moteurs surestimes de plus de 20 cycles ===")
    print(tableau_moteurs[tableau_moteurs["surestime_20"]].sort_values("ecart", ascending=False).round(1).to_string(index=False))
    print()

moteurs_surestimes_phase3 = set(tableaux_surestimes["XGBoost Phase 3"].query("surestime_20")["unit_number"])
moteurs_surestimes_retenu = set(tableaux_surestimes[NOM_RETENU].query("surestime_20")["unit_number"])
print(f"Moteurs surestimes par les deux modeles : {sorted(moteurs_surestimes_phase3 & moteurs_surestimes_retenu)}")
print(f"Seulement par Phase 3 : {sorted(moteurs_surestimes_phase3 - moteurs_surestimes_retenu)} ; "
      f"seulement par le retenu : {sorted(moteurs_surestimes_retenu - moteurs_surestimes_phase3)}")

### F.1 Tester des hypothèses plutôt que les affirmer

Deux hypothèses plausibles :

- **H1** : les moteurs surestimés ont des trajectoires observées **plus courtes** (le modèle a vu peu de
  cycles, le moteur « a l'air » encore neuf) ;
- **H2** : les moteurs surestimés ont des RUL vraies **différentes** des autres (plus petites ou plus grandes).

Outils : comparaison des médianes, et **test de Mann-Whitney** (deux groupes indépendants — surestimés vs
autres — comparés par les rangs, sans supposer de loi normale : « les valeurs d'un groupe ont-elles tendance
à être plus grandes que celles de l'autre ? »). **Limite majeure : n est petit** (une dizaine de moteurs
surestimés contre environ 90) : une p-valeur > 0,05 ne prouve pas l'absence d'effet, elle dit seulement que
ces données ne suffisent pas à le montrer. En complément, on utilise les **100 moteurs** : corrélation de
Spearman entre l'écart signé (prédiction − vérité) et chaque variable, qui ne dépend pas du seuil de 20 cycles.

In [ ]:
lignes = []
for nom_modele in ["XGBoost Phase 3", NOM_RETENU]:
    tableau_moteurs = tableaux_surestimes[nom_modele]
    groupe_surestimes = tableau_moteurs[tableau_moteurs["surestime_20"]]
    groupe_autres = tableau_moteurs[~tableau_moteurs["surestime_20"]]
    for variable in ["nb_cycles_observes", "rul_vraie"]:
        test_mw = stats.mannwhitneyu(groupe_surestimes[variable], groupe_autres[variable], alternative="two-sided")
        correlation = stats.spearmanr(tableau_moteurs["ecart"], tableau_moteurs[variable])
        ligne = {}
        ligne["modele"] = nom_modele
        ligne["variable"] = variable
        ligne["n_surestimes"] = len(groupe_surestimes)
        ligne["n_autres"] = len(groupe_autres)
        ligne["mediane_surestimes"] = groupe_surestimes[variable].median()
        ligne["mediane_autres"] = groupe_autres[variable].median()
        ligne["p_valeur_mann_whitney"] = test_mw.pvalue
        ligne["rho_spearman_ecart_100_moteurs"] = correlation.statistic
        ligne["p_valeur_spearman"] = correlation.pvalue
        lignes.append(ligne)
tableau_hypotheses = pd.DataFrame(lignes).set_index(["modele", "variable"])
tableau_hypotheses.round(4)

### F.2 Figure : écart de prédiction selon la trajectoire observée et selon la RUL vraie

Un point par moteur test et par modèle. Au-dessus de la ligne pointillée : surestimé de plus de 20 cycles.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6), sharey=True)
styles_modeles = [("XGBoost Phase 3", COULEUR_1, "o"), (NOM_RETENU, COULEUR_2, "^")]

for ax, variable, titre_x in [(axes[0], "nb_cycles_observes", "nombre de cycles observés dans test_FD001"),
                               (axes[1], "rul_vraie", "RUL vraie (RUL_FD001.txt)")]:
    for nom_modele, couleur, marqueur in styles_modeles:
        tableau_moteurs = tableaux_surestimes[nom_modele]
        ax.scatter(tableau_moteurs[variable], tableau_moteurs["ecart"], color=couleur, marker=marqueur, s=35,
                   alpha=0.7, label=nom_modele)
    ax.axhline(0, color="black", linewidth=1)
    ax.axhline(SEUIL_SURESTIMATION, color=COULEUR_TEXTE, linestyle=":", label=f"surestimation de {SEUIL_SURESTIMATION} cycles")
    ax.set_xlabel(titre_x)
axes[0].set_ylabel("écart = RUL prédite − RUL vraie (cycles) ; > 0 = surestimation")
axes[0].set_title("Écart vs longueur de la trajectoire observée")
axes[1].set_title("Écart vs RUL vraie")
axes[0].legend(fontsize=8, loc="lower right")
fig.suptitle("Test officiel FD001 — d'où viennent les surestimations ?")
plt.tight_layout()
plt.savefig(os.path.join(DOSSIER_FIGURES, "04_moteurs_surestimes.png"), dpi=150)
plt.show()

### F.3 Conclusion de la section F

- **Ce sont presque les mêmes moteurs** : 8 des 9 moteurs surestimés de plus de 20 cycles le sont par les
  deux modèles (15, 18, 27, 37, 41, 67, 72, 79) ; seuls le moteur 2 (Phase 3 seulement) et le moteur 50
  (retenu seulement) diffèrent. La pénalité réduit **l'ampleur** des pires surestimations (moteur 37 :
  +54 → +34 cycles ; moteur 15 : +34 → +27) mais pas leur **nombre**. Ces moteurs sont difficiles pour les
  deux modèles : la cause est plus probablement dans les données de ces moteurs que dans un réglage.
- **H1 (trajectoires observées plus courtes) : non démontrée.** Médiane de 121 cycles observés pour les
  surestimés contre 137 pour les autres, mais Mann-Whitney p = 0,07 (Phase 3) et 0,10 (retenu) ; sur les
  100 moteurs, corrélation écart / cycles observés ρ = −0,19 (p = 0,06) pour la Phase 3 et −0,06 (p = 0,55)
  pour le retenu. Au mieux une faible tendance, non significative avec si peu de moteurs. Les trois
  surestimations les plus dangereuses (moteurs 37, 18 et 41 : RUL vraie de 18 à 28 cycles, donc des alertes
  manquées) ont d'ailleurs des trajectoires de longueur ordinaire (121 à 133 cycles).
- **H2 (RUL vraie différente) : non démontrée entre les deux groupes** (médiane 63 contre 90 ; Mann-Whitney
  p = 0,12 et 0,07). La corrélation sur les 100 moteurs entre l'écart et la RUL vraie est forte (ρ = −0,34 et
  −0,54, p < 0,001), mais elle est en grande partie **mécanique** : le modèle ne prédit jamais au-delà de
  125, donc les moteurs à grande RUL vraie sont forcément sous-estimés, et un modèle imparfait ramène ses
  prédictions vers le milieu de la plage. Fait observable : aucun moteur dont la RUL vraie dépasse 98 cycles
  n'est surestimé de plus de 20 cycles (panneau de droite).
- **Piste, non testée ici** : examiner les capteurs des moteurs 37, 18 et 41 pour comprendre pourquoi ils
  « ont l'air » plus sains qu'ils ne le sont — ce sont eux qui portent les alertes manquées qui comptent.

## G. Synthèse et fichiers de résultats

### G.1 Tableau final

Validation croisée (mêmes 5 plis, sert au choix) et test officiel (évalué une fois, après le choix), pour
les modèles candidats.

In [ ]:
resultats_cv_finaux = {}  # nom -> resultats de validation croisee des modeles evalues sur le test
for nom_modele in ["XGBoost Phase 3", "Random Forest", "LightGBM"]:
    resultats_cv_finaux[nom_modele] = resultats_cv_base[nom_modele]
for nom_candidat in resultats_cv_optuna:
    resultats_cv_finaux[nom_candidat] = resultats_cv_optuna[nom_candidat]

lignes = []
for nom_modele in resultats_test:
    ligne_cv = resumer_cv(nom_modele, resultats_cv_finaux[nom_modele], NB_MOTEURS_TRAIN)
    ligne_test = resumer_test(nom_modele, resultats_test[nom_modele])
    ligne = {}
    ligne["modele"] = nom_modele
    ligne["rmse_cv"] = ligne_cv["rmse_cv_moyen"]
    ligne["nasa_cv"] = ligne_cv["nasa_cv_moyen"]
    ligne["surestimes_oof"] = ligne_cv["surestimes_oof"]
    ligne["rmse_test"] = ligne_test["rmse_test"]
    ligne["rmse_test_plafonnee"] = ligne_test["rmse_test_rul_plafonnee"]
    ligne["nasa_test"] = ligne_test["nasa_test"]
    ligne["surestimes_test"] = ligne_test["surestimes_test"]
    ligne["alertes_manquees_test"] = ligne_test["alertes_manquees_test"]
    ligne["fausses_alertes_test"] = ligne_test["fausses_alertes_test"]
    lignes.append(ligne)
tableau_synthese = pd.DataFrame(lignes).set_index("modele")
tableau_synthese.round(2)

### G.2 Écriture des résultats

- `docs/resultats/04_comparaison_cv.csv` : tous les modèles évalués en validation croisée dans ce notebook
  (section B, grille de la section C, candidats Optuna), une ligne par modèle ;
- `docs/resultats/04_comparaison_test.csv` : les modèles évalués sur le test (section E) ;
- `docs/resultats/04_config_retenue.json` : la configuration retenue, écrite par `sauvegarder_config`
  (validée avant écriture) ; c'est ce fichier que lit la démo.

In [ ]:
# 1) validation croisee : sections B, C et D
lignes = []
for nom_modele in resultats_cv_base:
    ligne = resumer_cv(nom_modele, resultats_cv_base[nom_modele], NB_MOTEURS_TRAIN)
    ligne["section"] = "B - reglages de base"
    lignes.append(ligne)
for reglage in reglages_penalite:
    ligne = resumer_cv(reglage["reglage"], resultats_cv_penalite[reglage["reglage"]], NB_MOTEURS_TRAIN)
    ligne["section"] = "C - penalite de surestimation"
    lignes.append(ligne)
for nom_candidat in resultats_cv_optuna:
    ligne = resumer_cv(nom_candidat, resultats_cv_optuna[nom_candidat], NB_MOTEURS_TRAIN)
    ligne["section"] = "D - candidat Optuna"
    lignes.append(ligne)
tableau_comparaison_cv = pd.DataFrame(lignes)
chemin_cv = os.path.join(DOSSIER_RESULTATS, "04_comparaison_cv.csv")
tableau_comparaison_cv.to_csv(chemin_cv, index=False)

# 2) test officiel : section E (+ moyenne et ecart-type sur 5 graines pour Phase 3 et le retenu)
tableau_comparaison_test = tableau_test.reset_index()
for nom_modele in ["XGBoost Phase 3", NOM_RETENU]:
    masque = tableau_comparaison_test["modele"] == nom_modele
    resultats_graines = tableau_graines[tableau_graines["modele"] == nom_modele]
    tableau_comparaison_test.loc[masque, "rmse_test_moyenne_5_graines"] = resultats_graines["rmse_test"].mean()
    tableau_comparaison_test.loc[masque, "rmse_test_ecart_type_5_graines"] = resultats_graines["rmse_test"].std()
    tableau_comparaison_test.loc[masque, "nasa_test_moyenne_5_graines"] = resultats_graines["nasa_test"].mean()
    tableau_comparaison_test.loc[masque, "nasa_test_ecart_type_5_graines"] = resultats_graines["nasa_test"].std()
chemin_test = os.path.join(DOSSIER_RESULTATS, "04_comparaison_test.csv")
tableau_comparaison_test.to_csv(chemin_test, index=False)

# 3) configuration retenue (schema de src/config_modele.py), lue par la demo
resultats_cv_retenu = resultats_cv_optuna[NOM_RETENU]
resultats_test_retenu = resultats_test[NOM_RETENU]
resultats_graines_retenu = tableau_graines[tableau_graines["modele"] == NOM_RETENU]

metriques_cv = {}
metriques_cv["rmse_cv_moyen"] = resultats_cv_retenu["rmse_moyen"]
metriques_cv["rmse_cv_ecart_type"] = resultats_cv_retenu["rmse_ecart_type"]
metriques_cv["mae_cv_moyen"] = resultats_cv_retenu["mae_moyen"]
metriques_cv["nasa_cv_moyen"] = resultats_cv_retenu["nasa_moyen"]
metriques_cv["surestimes_oof"] = resultats_cv_retenu["surestimes_oof"]
metriques_cv["alertes_manquees_oof"] = resultats_cv_retenu["alertes_manquees_oof"]
metriques_cv["fausses_alertes_oof"] = resultats_cv_retenu["fausses_alertes_oof"]
metriques_cv["biais_oof"] = resultats_cv_retenu["biais_oof"]

metriques_test = {}
metriques_test["rmse"] = resultats_test_retenu["rmse"]
metriques_test["rmse_contre_rul_plafonnee"] = resultats_test_retenu["rmse_contre_rul_plafonnee"]
metriques_test["mae"] = resultats_test_retenu["mae"]
metriques_test["score_nasa"] = resultats_test_retenu["score_nasa"]
metriques_test["nb_surestimes"] = resultats_test_retenu["nb_surestimes"]
metriques_test["alertes_manquees"] = resultats_test_retenu["alertes_manquees"]
metriques_test["fausses_alertes"] = resultats_test_retenu["fausses_alertes"]
metriques_test["rmse_moyenne_5_graines"] = resultats_graines_retenu["rmse_test"].mean()
metriques_test["rmse_ecart_type_5_graines"] = resultats_graines_retenu["rmse_test"].std()

config_a_ecrire = copy.deepcopy(config_retenue)
config_a_ecrire["nom"] = NOM_RETENU
config_a_ecrire["regle_de_choix"] = TEXTE_REGLE_DE_CHOIX
config_a_ecrire["metriques_cv"] = metriques_cv
config_a_ecrire["metriques_test"] = metriques_test
config_a_ecrire["source"] = (f"notebooks/04_comparaison_evaluation.ipynb ; essai {int(essai_retenu['numero'])} de "
                             f"docs/resultats/04_optuna_{essai_retenu['etude']}.csv")
config_a_ecrire["date"] = datetime.date.today().isoformat()
chemin_config = sauvegarder_config(config_a_ecrire)

print("Fichiers ecrits :")
print(" ", os.path.relpath(chemin_cv))
print(" ", os.path.relpath(chemin_test))
print(" ", os.path.relpath(chemin_config), "->", decrire_config(config_a_ecrire))

### G.3 Ce qui est prouvé, ce qui ne l'est pas, ce qui reste à faire

**Prouvé (validation croisée, mêmes 5 plis, tests appariés sur 100 moteurs)**

- Les trois ensembles d'arbres battent largement la référence naïve (RMSE CV ≈ 16,9 contre 41,7) et sont
  **à égalité entre eux** à réglages de base (p > 0,5). XGBoost se justifie par sa vitesse et par sa perte
  sur mesure, pas par sa précision.
- Changer les constantes 13/10 du score ne change aucune prédiction (démonstration exacte).
- Pénaliser la surestimation **dans l'apprentissage** réduit les surestimations, avec un coût mesuré en RMSE,
  en fausses alertes et en biais (grille de la section C, monotone en α).
- Le modèle retenu réduit les surestimations OOF > 20 de 36 % (p < 0,0001) et le score NASA CV de 29 %
  (p = 0,007 / 0,028, fragilisé par l'effet de sélection), sans différence détectable de RMSE par moteur ;
  en échange de 47 % de fausses alertes en plus.
- Tout est reproductible : le JSON reconstruit le modèle au bit près (contrôle D.6), les études Optuna
  redonnent les mêmes essais à graine égale (`tests/test_optimisation.py`), les chiffres de la Phase 3 sont
  retrouvés à l'identique.

**Non concluant**

- Sur le test officiel (100 points), aucun modèle n'est significativement différent de XGBoost Phase 3. La
  baisse du score NASA du retenu (853 → 675) vient d'un seul moteur (n°37).
- Les alertes manquées du test (7) ne changent avec aucun modèle.
- Les hypothèses sur les moteurs surestimés (trajectoires courtes, RUL vraie) ne sont pas démontrées : trop
  peu de moteurs.

**À faire, limites assumées**

- Modèles séquentiels (LSTM, CNN 1D) : pas encore faits.
- Sous-ensembles FD002 à FD004 : pas encore faits (FD001 seul : une condition de vol, un mode de panne).
- Le seuil d'alerte de 30 cycles et le seuil de surestimation de 20 cycles sont des hypothèses de travail, à
  discuter.
- La validation croisée évalue tous les cycles, alors que le test n'évalue que le dernier cycle d'une
  trajectoire tronquée : un protocole de validation qui imite cette troncature serait plus fidèle (déjà noté
  en Phase 3).
- Une seule graine Optuna (42) et des budgets d'essais inégaux (LightGBM : 40) ; la tolérance de 5 % de la
  règle de choix est un choix d'équipe, pas une valeur tirée des données.
- Examiner les capteurs des moteurs 37, 18 et 41 (alertes manquées communes à tous les modèles).

In [ ]:
duree_notebook = time.perf_counter() - debut_notebook
print(f"Notebook execute en {duree_notebook / 60:.1f} minutes.")